## Imports

In [1]:
import re
from collections import defaultdict

import numpy as np
import pandas as pd
from lemminflect import getAllLemmas, getAllLemmasOOV
from nltk import pos_tag, word_tokenize, WordNetLemmatizer
from num2words import num2words as _num2words
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

from analysis_helpers import Episode, Participant, show_content_warning
from analysis_helpers.constants import (
    ENDFRAME_TIMES, 
    LEMMATIZER_EXCLUSIONS,
    PROCESSED_DIR, 
    STOP_WORDS,
    TEXT_SUBSTITUTIONS
)

In [2]:
show_content_warning()

⚠️ The episodes of [*Atlanta*](https://en.wikipedia.org/wiki/Atlanta_(TV_series)) viewed by participants in this study explore themes of racism, homophobia, and other forms of discrimination. Consequently, certain files in this repository&mdash;possibly including this one&mdash;contain references to language that may be offensive or harmful. This language appears only in service of accurately representing and analyzing the stimuli and participants' responses, and its inclusion does not reflect an endorsement of its use by the authors.

## Functions

In [3]:
def num2words(number: str) -> str:
    if len(number) == 4:
        return _num2words(number, to='year')
    return _num2words(number, to='cardinal')

In [4]:
# POS tag mapping, format: {Treebank tag (1st letter only): Wordnet}
POS_MAPPING = defaultdict(
    lambda: 'n',     # defaults to noun
    {
        'N': 'n',    # noun types
        'P': 'n',    # pronoun types, predeterminers
        'V': 'v',    # verb types
        'J': 'a',    # adjective types
        'D': 'a',    # determiner
        'R': 'r'     # adverb types
    }
)

wordnet_lemmatizer = WordNetLemmatizer()

In [83]:
from collections import Counter


def preprocess_text(text: str) -> str:
    # combine mult-word tokens, standardize names, replace euphemisms, 
    # etc. before tokenizing/lemmatizing
    for pattern, replacement in TEXT_SUBSTITUTIONS.items():
        text = pattern.sub(replacement, text)
    
    # convert digits to word form
    text = re.sub(r'(\d+)', lambda match: num2words(match.group(0)), text)
    
    # lemmatize
    lemmas = []
    # temporarily replace asterisks in censored expletives with tildes 
    # so the tokenizer doesn't split on them
    text = text.replace('*', '~')
    for word, tag in pos_tag(word_tokenize(text)):
        if word.lower() not in STOP_WORDS:
            lemma = lemmatize_lemminflect(word, tag).lower()
            if lemma not in STOP_WORDS:
                lemmas.append(lemma_with_retag)
                
    text = ' '.join(lemmas).replace('~', '*')
    

    return text

In [6]:
def transform_episode(annotations: pd.DataFrame):
    # drop frame, onset time, (manually labeled) scene name, keep rest
    text_cols = annotations.loc[:, 'Narrative details (external events)':'Setting']
    # concatenate all annotated features for each shot. Adding period as
    # delimiter helps POS tagger
    joined = text_cols.apply(lambda row: '. '.join(row.dropna()), axis=1)
    # remove period delimiter if annotation text already ended with 
    # punctuation
    joined = joined.str.replace(r'(?<=\W)\. ', ' ', regex=True)
    # remove single quotes (used for nested quotes in "Speech" column)
    joined = joined.str.replace(r"(?<=\W)'|(?<!s)'(?=\W)", '', regex=True)
    
    
    processed_annot = joined.apply(preprocess_text)


## Load data

In [7]:
atlep1 = Episode('atlep1')
atlep2 = Episode('atlep2')
arrdev = Episode('arrdev')

participants = Participant.load_all()

In [114]:
TREEBANK_LEMMINFLECT_MAPPING = {
    # lemminflect accepts only a subset of the Universal Dependencies 
    # tagset: NOUN, PROPN, VERB, ADJ, ADV, AUX
    'JJ': 'ADJ',
    'JJR': 'ADJ',
    'JJS': 'ADJ',
    'MD': 'AUX',    # note: might wanna make this "VERB" if issues?
    'NN': 'NOUN',
    'NNS': 'NOUN',
    'NNP': 'PROPN',
    'NNPS': 'PROPN',
    'RB': 'ADV',
    'RBR': 'ADV',
    'RBS': 'ADV',
    'VB': 'VERB',
    'VBD': 'VERB',
    'VBG': 'VERB',
    'VBN': 'VERB',
    'VBP': 'VERB',
    'VBZ': 'VERB',    
}


def lemmatize(word: str, treebank_tag: str) -> str:
    if (
            treebank_tag in {
                '.', ',', "''", '``', ':', '(', ')', # punctuation
                'CC',    # coordinating conjunction, always stopwords
                'CD',    # cardinal number, not lemmatizeable
                'EX',    # "existential 'there'", always literal "there"
                'FW',    # foreign word (really mis-tagged tokens)
                'POS',   # possessive ending
                'RP',    # particle
                'TO',    # literal "to"
                'UH',    # interjection
            }
            or '_' in word    # n-grams from TEXT_SUBSTITUTIONS
            or '~' in word    # censored profanity
            or word.lower() in LEMMATIZER_EXCLUSIONS
    ):
        return word
    
    universal_tag = TREEBANK_LEMMINFLECT_MAPPING.get(treebank_tag)
    # returns a dict of {POS: (lemmas, ...), ...}
    lemma_candidates = getAllLemmas(word, universal_tag)
    
    if len(lemma_candidates) == 0:
        # no lemmas found via dictionary lookup.
        # Usually indicates word was mis-tagged because sentence 
        # fragments in annotations & stream-of-consciousness in recall 
        # transcripts both confuse the POS tagger. Try re-tagging first.
        new_treebank_tag = pos_tag([word])[0][1]
        if new_treebank_tag != treebank_tag:
            return lemmatize(word, new_treebank_tag)
        
        # If we have a POS tag, also try rule-based lemmatization
        if universal_tag is not None:
            # returns a 1-item dict of {POS: (lemma,)}, or an empty dict
            lemma_candidates_oov = getAllLemmasOOV(word, universal_tag)
            if len(lemma_candidates_oov) == 0:
                return word
            
            lemma_oov = lemma_candidates_oov[universal_tag][0]
            if lemma_oov != word:
                if lemma_oov.endswith('z') and word[word.rindex('z') + 1] == 'e':
                    # when inflections of -ze verbs (e.g., "recognized", 
                    # "realizes", etc.) are mis-tagged as different POS, 
                    # rule-based lemmatizer removes trailing "e"
                    lemma_oov = f'{lemma_oov}e'
                    
            return lemma_oov
        return word
    
    if len(lemma_candidates) > 1:
        # universal_tag is None and word has lemmas for multiple POS
        if any(len(pos_lemmas) > 1 for pos_lemmas in lemma_candidates.values()):
            # word has >1 possible lemma for at least 1 POS
            raise RuntimeError(
                f'Multiple potential lemmas found:\n\t{word=}, '
                f'{treebank_tag=}, {universal_tag=}, {lemma_candidates=}'
            )
        
        unique_lemmas = tuple(set(pos_lemmas[0] for pos_lemmas in lemma_candidates.values()))
        if len(unique_lemmas) > 1:
            # word has different possible lemmas for different POS.
            # Default to choosing the shortest one
            if any(len(lem) == len(unique_lemmas[0]) for lem in unique_lemmas[1:]):
                # multiple possible lemmas are the same length
                raise RuntimeError(
                    f'Multiple potential lemmas found:\n\t{word=}, '
                    f'{treebank_tag=}, {universal_tag=}, {lemma_candidates=}'
                )
            return min(unique_lemmas, key=len)
            
        return unique_lemmas[0]
    
    # Note: checked all instances in episode annotations where >1 lemma 
    # found for given word/POS, and 1st option is always the "right" one
    return tuple(lemma_candidates.values())[0][0]

In [84]:
all_annot = pd.concat((
    atlep1.annotations.loc[:, 'Narrative details (external events)':'Setting'].apply(lambda row: '. '.join(row.dropna()), axis=1),
    atlep2.annotations.loc[:, 'Narrative details (external events)':'Setting'].apply(lambda row: '. '.join(row.dropna()), axis=1),
    arrdev.annotations.loc[:, 'Narrative details (external events)':'Setting'].apply(lambda row: '. '.join(row.dropna()), axis=1)
)).reset_index(drop=True)
joined = all_annot.str.replace(r'(?<=\W)\. ', ' ', regex=True)
joined = joined.str.replace(r"(?<=\W)'|(?<!s)'(?=\W)", '', regex=True)


EPISODE_LEMMATIZATIONS = defaultdict(Counter)
processed_annot = joined.apply(preprocess_text, lemmatization_counter=EPISODE_LEMMATIZATIONS)

lemmatized via getAllLemmasOOV: ('confronts', 'NOUN') -> confront
lemmatized via getAllLemmasOOV: ('confronts', 'NOUN') -> confront
lemmatized via getAllLemmasOOV: ('relaxed', 'ADV') -> relax
retagging changed lemma for relaxed. Without: (relaxed, ADV) -> relax, With: (relaxed, NOUN) -> relaxed
lemmatized via getAllLemmasOOV: ('mocks', 'NOUN') -> mock
lemmatized via getAllLemmasOOV: ('mocks', 'NOUN') -> mock
retagging changed lemma for stunting. Without: (stunting, NOUN) -> stunting, With: (stunting, VERB) -> stunt
lemmatized via getAllLemmasOOV: ('interjects', 'NOUN') -> interject
lemmatized via getAllLemmasOOV: ('interjects', 'NOUN') -> interject
lemmatized via getAllLemmasOOV: ('surprised', 'ADJ') -> surprise
lemmatized via getAllLemmasOOV: ('surprised', 'ADJ') -> surprise
lemmatized via getAllLemmasOOV: ('mocks', 'NOUN') -> mock
lemmatized via getAllLemmasOOV: ('mocks', 'NOUN') -> mock
lemmatized via getAllLemmasOOV: ('Worldstar', 'VERB') -> Worldsta
retagging changed lemma for Wor

lemmatized via getAllLemmasOOV: ('tells', 'NOUN') -> tell
lemmatized via getAllLemmasOOV: ('tells', 'NOUN') -> tell
lemmatized via getAllLemmasOOV: ('tells', 'NOUN') -> tell
lemmatized via getAllLemmasOOV: ('pissed', 'ADJ') -> piss
lemmatized via getAllLemmasOOV: ('introduces', 'NOUN') -> introduce
lemmatized via getAllLemmasOOV: ('introduces', 'NOUN') -> introduce
lemmatized via getAllLemmasOOV: ('feigns', 'ADJ') -> feign
lemmatized via getAllLemmasOOV: ('feigns', 'NOUN') -> feign
lemmatized via getAllLemmasOOV: ('disinterested', 'VERB') -> disinterest
lemmatized via getAllLemmasOOV: ('disinterested', 'VERB') -> disinterest
lemmatized via getAllLemmasOOV: ('knows', 'NOUN') -> know
lemmatized via getAllLemmasOOV: ('knows', 'NOUN') -> know
lemmatized via getAllLemmasOOV: ('straight-faced', 'ADJ') -> straight-face
lemmatized via getAllLemmasOOV: ('straight-faced', 'ADJ') -> straight-face
lemmatized via getAllLemmasOOV: ('unimpressed', 'VERB') -> unimpress
retagging changed lemma for unim

lemmatized via getAllLemmasOOV: ('PJ', 'VERB') -> P
retagging changed lemma for PJ. Without: (PJ, VERB) -> p, With: (PJ, NOUN) -> pj
lemmatized via getAllLemmasOOV: ('Crickets', 'PROPN') -> Cricket
lemmatized via getAllLemmasOOV: ('open-mouthed', 'ADJ') -> open-mouth
lemmatized via getAllLemmasOOV: ('open-mouthed', 'ADJ') -> open-mouth
lemmatized via getAllLemmasOOV: ('Crickets', 'PROPN') -> Cricket
lemmatized via getAllLemmasOOV: ('PJ', 'VERB') -> P
retagging changed lemma for PJ. Without: (PJ, VERB) -> p, With: (PJ, NOUN) -> pj
lemmatized via getAllLemmasOOV: ('rappers', 'NOUN') -> rapper
lemmatized via getAllLemmasOOV: ('rappers', 'NOUN') -> rapper
lemmatized via getAllLemmasOOV: ('PJ', 'VERB') -> P
retagging changed lemma for PJ. Without: (PJ, VERB) -> p, With: (PJ, NOUN) -> pj
lemmatized via getAllLemmasOOV: ('rappers', 'NOUN') -> rapper
lemmatized via getAllLemmasOOV: ('rappers', 'NOUN') -> rapper
lemmatized via getAllLemmasOOV: ('PJ', 'VERB') -> P
retagging changed lemma for PJ.

retagging changed lemma for screaming. Without: (screaming, NOUN) -> screaming, With: (screaming, VERB) -> scream
lemmatized via getAllLemmasOOV: ('tells', 'NOUN') -> tell
lemmatized via getAllLemmasOOV: ('tells', 'NOUN') -> tell
lemmatized via getAllLemmasOOV: ('sits', 'NOUN') -> sit
lemmatized via getAllLemmasOOV: ('sits', 'NOUN') -> sit
lemmatized via getAllLemmasOOV: ('disinterested', 'VERB') -> disinterest
lemmatized via getAllLemmasOOV: ('disinterested', 'VERB') -> disinterest
lemmatized via getAllLemmasOOV: ('slouches', 'NOUN') -> slouch
lemmatized via getAllLemmasOOV: ('slouches', 'NOUN') -> slouch
lemmatized via getAllLemmasOOV: ('feels', 'ADJ') -> feel
lemmatized via getAllLemmasOOV: ('feels', 'NOUN') -> feel
lemmatized via getAllLemmasOOV: ('unmotivated', 'ADJ') -> unmotivate
lemmatized via getAllLemmasOOV: ('unmotivated', 'ADJ') -> unmotivate
lemmatized via getAllLemmasOOV: ('disinterested', 'VERB') -> disinterest
lemmatized via getAllLemmasOOV: ('disinterested', 'VERB') ->

lemmatized via getAllLemmasOOV: ('Thanks', 'NOUN') -> Thank
lemmatized via getAllLemmasOOV: ('Thanks', 'NOUN') -> Thank
lemmatized via getAllLemmasOOV: ('falters', 'NOUN') -> falter
lemmatized via getAllLemmasOOV: ('falters', 'NOUN') -> falter
lemmatized via getAllLemmasOOV: ('high-functioning', 'ADJ') -> high-function
retagging changed lemma for high-functioning. Without: (high-functioning, ADJ) -> high-function, With: (high-functioning, NOUN) -> high-functioning
lemmatized via getAllLemmasOOV: ('responds', 'NOUN') -> respond
lemmatized via getAllLemmasOOV: ('responds', 'NOUN') -> respond
lemmatized via getAllLemmasOOV: ('sits', 'NOUN') -> sit
lemmatized via getAllLemmasOOV: ('sits', 'NOUN') -> sit
lemmatized via getAllLemmasOOV: ('guys', 'ADJ') -> guy
retagging changed lemma for selling. Without: (selling, NOUN) -> selling, With: (selling, VERB) -> sell
lemmatized via getAllLemmasOOV: ('tells', 'NOUN') -> tell
lemmatized via getAllLemmasOOV: ('tells', 'NOUN') -> tell
lemmatized via g

lemmatized via getAllLemmasOOV: ('tells', 'NOUN') -> tell
lemmatized via getAllLemmasOOV: ('tells', 'NOUN') -> tell
lemmatized via getAllLemmasOOV: ('Bluths', 'PROPN') -> Bluth
lemmatized via getAllLemmasOOV: ('Bluths', 'NOUN') -> Bluth
lemmatized via getAllLemmasOOV: ('nods', 'NOUN') -> nod
lemmatized via getAllLemmasOOV: ('nods', 'NOUN') -> nod
retagging changed lemma for quitting. Without: (quitting, NOUN) -> quitting, With: (quitting, VERB) -> quit
lemmatized via getAllLemmasOOV: ('surprised', 'ADJ') -> surprise
lemmatized via getAllLemmasOOV: ('surprised', 'ADJ') -> surprise
lemmatized via getAllLemmasOOV: ('Securities', 'PROPN') -> Security
lemmatized via getAllLemmasOOV: ('jumpsuits', 'NOUN') -> jumpsuit
lemmatized via getAllLemmasOOV: ('jumpsuits', 'NOUN') -> jumpsuit
lemmatized via getAllLemmasOOV: ('beckons', 'NOUN') -> beckon
lemmatized via getAllLemmasOOV: ('beckons', 'NOUN') -> beckon
lemmatized via getAllLemmasOOV: ('nods', 'NOUN') -> nod
lemmatized via getAllLemmasOOV: (

## WITHOUT RETAGGING:

In [10]:
print(f'{len(tuple(v for v in EPISODE_LEMMATIZATIONS.values() if len(v) > 1))} inconsistencies (out of {len(EPISODE_LEMMATIZATIONS)}):\n')
for word, lemma_counter in EPISODE_LEMMATIZATIONS.items():
    if len(lemma_counter) > 1:
        print(f'\x1b[1m{word}\x1b[0m')
        for lemma, count in lemma_counter.items():
            print(f'    {lemma}    {count}')
        print('======================')

73 inconsistencies (out of 3045):

worried
    worry    12
    worried    3
speaking
    speak    13
    speaking    1
chest
    chest    6
    ch    1
worldstar
    worldstar    1
    worldsta    1
broke
    break    2
    broke    2
stunting
    stunt    1
    stunting    1
street
    street    12
    stre    2
confused
    confuse    24
    confused    2
frustrated
    frustrate    25
    frustrated    1
waiting
    wait    240
    waiting    5
playing
    play    33
    playing    6
listening
    listen    6
    listening    1
tired
    tire    7
    tired    3
seaweed
    seaweed    2
    seawee    1
..
    ..    7
    .    2
meaning
    meaning    2
    mean    2
disappointed
    disappointed    6
    disappoint    14
joking
    joke    4
    joking    1
kissing
    kiss    7
    kissing    3
lives
    live    4
    life    3
better
    good    7
    well    7
missing
    miss    3
    missing    2
living
    live    2
    living    1
selling
    sell    4
    selling    1
watchi

## WITH RETAGGING:

In [10]:
print(f'{len(tuple(v for v in EPISODE_LEMMATIZATIONS.values() if len(v) > 1))} inconsistencies (out of {len(EPISODE_LEMMATIZATIONS)}):\n')
for word, lemma_counter in EPISODE_LEMMATIZATIONS.items():
    if len(lemma_counter) > 1:
        print(f'\x1b[1m{word}\x1b[0m')
        for lemma, count in lemma_counter.items():
            print(f'    {lemma}    {count}')
        print('======================')

41 inconsistencies (out of 3045):

worried
    worry    12
    worried    3
relaxed
    relaxed    1
    relax    3
broke
    break    2
    broke    2
confused
    confuse    24
    confused    2
frustrated
    frustrate    25
    frustrated    1
annoyed
    annoy    29
    annoyed    3
tired
    tire    7
    tired    3
meaning
    meaning    2
    mean    2
unappealing
    unappeal    1
    unappealing    1
interesting
    interesting    1
    interest    1
disappointed
    disappointed    6
    disappoint    14
joking
    joke    4
    joking    1
lives
    live    4
    life    3
better
    good    7
    well    7
missing
    miss    3
    missing    2
living
    live    2
    living    1
amazing
    amaze    1
    amazing    2
later
    later    5
    late    1
drinking
    drink    3
    drinking    2
seen
    see    12
    seen    1
met
    meet    3
    met    1
acting
    act    4
    acting    2
caught
    catch    5
    caught    1
closer
    closer    1
    close    1
mean

In [72]:
def find_lemma_instance(text: str, target_word: str, target_lemma=None) -> str:
    for pattern, replacement in TEXT_SUBSTITUTIONS.items():
        text = pattern.sub(replacement, text)

    text = re.sub(r'(\d+)', lambda match: num2words(match.group(0)), text)
    
    text = text.replace('*', '~')
    for word, tag in pos_tag(word_tokenize(text)):
        if word.lower() == target_word:
            lemma = lemmatize_lemminflect(word, tag, try_retag=True).lower()
            
            if target_lemma is None or lemma == target_lemma:
                print(text)
                print(f'{word} -> {lemma}')
                print('================================================')

In [263]:
joined.apply(find_lemma_instance, target_word='shoot', target_lemma=None)
pass

Darius, sitting on the couch, suggests they go play pool instead, since the people there are cool. Darius. no. Well, yo, maybe we could go shoot some pool, right? The cool up there. Darius. indoor. Alfred_Paper_Boy's house
shoot -> shoot
Asia runs, chased by a young black boy, Demarrio, wearing a hat and holding a toy gun and pretending to shoot her. Asia pretends to be shot and falls to the ground. Asia, Demarrio. no. outdoor. Outside Decatur Apartments
shoot -> shoot
Demarrio turns and points the toy gun at Chris. He tells him to back up or he will shoot. Demarrio, Chris, Asia. no. Back up or I smoke you, man. Demarrio. outdoor. Outside Decatur Apartments
shoot -> shoot


In [22]:
def find_in_recalls(target, preprocess=False, show_subids=False, pad=50, lemmatization_counter=None):
    n_matches = 0
    
    if show_subids:
        episode_subids_substrs = {}
        for episode in ('atlep1', 'atlep2', 'arrdev'):
            subids_substrs = {}
            for p in participants:
                if (
                    (episode == 'atlep2' and p.condition == 'B') or 
                    (episode == 'arrdev' and p.condition == 'A')
                ):
                    continue
                participant_substrs = []
                p_transcript = p.transcripts[episode]
                if preprocess:
                    p_transcript = preprocess_text(p_transcript, lemmatization_counter=lemmatization_counter)
                for match in re.finditer(target, p_transcript, flags=re.IGNORECASE):
                    participant_substrs.append(p_transcript[match.start()-pad:match.start()+pad])
                
                if len(participant_substrs) > 0:
                    subids_substrs[p.subid] = participant_substrs
                    n_matches += len(participant_substrs)
            
            if len(subids_substrs) > 0:
                episode_subids_substrs[episode] = subids_substrs
        
        print(f'{n_matches} matches:\n')
        for episode, subids_substrs in episode_subids_substrs.items():
            print(f'{"="*(2*pad+8)}\n{episode}')
            for subid, substrs in subids_substrs.items():
                print(f'    {subid}')
                for substr in substrs:
                    print(f'        {substr}')

    else:
        all_recalls = (
            ' '.join([p.transcripts['atlep1'] for p in participants])
            + ' '.join([p.transcripts['atlep2'] for p in participants if p.condition == 'A'])
            + ' '.join([p.transcripts['arrdev'] for p in participants if p.condition == 'B'])
        )
        if preprocess:
            all_recalls = preprocess_text(all_recalls, lemmatization_counter=lemmatization_counter)
        substrs = []
        for match in re.finditer(target, all_recalls, flags=re.IGNORECASE):
            n_matches += 1
            substrs.append(all_recalls[match.start()-pad:match.start()+pad])

        print(f'{n_matches} matches:\n')
        for substr in substrs:        
            print(substr)

In [500]:
find_in_recalls(r'hearted', preprocess=False, show_subids=True)

1 matches:

arrdev
    MD-022819-B-02
        only care about money but she's particularly cold hearted his older brother's a magician in this all


In [497]:
target = r"robert"

# joined_annots = ' '.join(processed_annot)
joined_annots = ' '.join(joined)

pad = 50

substrs = []
n_matches = 0

for match in re.finditer(target, joined_annots, flags=re.IGNORECASE):
    n_matches += 1
    substrs.append(joined_annots[match.start()-pad:match.start()+pad])
    
print(f'{n_matches} matches:\n')
for substr in substrs:
    print(substr)

0 matches:



In [115]:
RECALL_LEMMATIZATIONS = defaultdict(Counter)
find_in_recalls(r'\bnope', preprocess=True, show_subids=True, lemmatization_counter=RECALL_LEMMATIZATIONS)

lemmatized via getAllLemmasOOV: ('seaweed', 'VERB') -> seawee
retagging changed lemma for seaweed. Without: (seaweed, VERB) -> seawee, With: (seaweed, NOUN) -> seaweed
lemmatized via getAllLemmasOOV: ('hundred', 'VERB') -> hundre
lemmatized via getAllLemmasOOV: ('hundred', 'VERB') -> hundre
lemmatized via getAllLemmasOOV: ('whatever', 'ADV') -> whatev
retagging changed lemma for whatever. Without: (whatever, ADV) -> whatev, With: (whatever, None) -> whatever
lemmatized via getAllLemmasOOV: ('foreshadowing', 'ADJ') -> foreshadow
retagging changed lemma for wearing. Without: (wearing, NOUN) -> wearing, With: (wearing, VERB) -> wear
retagging changed lemma for losing. Without: (losing, NOUN) -> losing, With: (losing, VERB) -> lose
lemmatized via getAllLemmasOOV: ('sirens', 'VERB') -> siren
lemmatized via getAllLemmasOOV: ('sirens', 'VERB') -> siren
lemmatized via getAllLemmasOOV: ('tiara', 'NOUN') -> tiarum
lemmatized via getAllLemmasOOV: ('tiara', 'NOUN') -> tiarum
lemmatized via getAllL

retagging changed lemma for speaking. Without: (speaking, NOUN) -> speaking, With: (speaking, VERB) -> speak
lemmatized via getAllLemmasOOV: ('texting', 'VERB') -> text
lemmatized via getAllLemmasOOV: ('texting', 'VERB') -> text
lemmatized via getAllLemmasOOV: ('chobani', 'ADJ') -> chobanus
lemmatized via getAllLemmasOOV: ('chobani', 'NOUN') -> chobanus
lemmatized via getAllLemmasOOV: ('tiara', 'NOUN') -> tiarum
lemmatized via getAllLemmasOOV: ('tiara', 'NOUN') -> tiarum
lemmatized via getAllLemmasOOV: ('starbucks', 'NOUN') -> starbuck
lemmatized via getAllLemmasOOV: ('starbucks', 'NOUN') -> starbuck
lemmatized via getAllLemmasOOV: ('asks', 'NOUN') -> ask
lemmatized via getAllLemmasOOV: ('asks', 'NOUN') -> ask
lemmatized via getAllLemmasOOV: ('asks', 'NOUN') -> ask
lemmatized via getAllLemmasOOV: ('asks', 'NOUN') -> ask
lemmatized via getAllLemmasOOV: ('towards', 'VERB') -> toward
lemmatized via getAllLemmasOOV: ('towards', 'NOUN') -> toward
lemmatized via getAllLemmasOOV: ('name-drops

lemmatized via getAllLemmasOOV: ('damaged', 'ADJ') -> damage
lemmatized via getAllLemmasOOV: ('tells', 'NOUN') -> tell
lemmatized via getAllLemmasOOV: ('tells', 'NOUN') -> tell
lemmatized via getAllLemmasOOV: ('tells', 'NOUN') -> tell
lemmatized via getAllLemmasOOV: ('tells', 'NOUN') -> tell
lemmatized via getAllLemmasOOV: ('tells', 'NOUN') -> tell
lemmatized via getAllLemmasOOV: ('tells', 'NOUN') -> tell
lemmatized via getAllLemmasOOV: ('tells', 'NOUN') -> tell
lemmatized via getAllLemmasOOV: ('tells', 'NOUN') -> tell
lemmatized via getAllLemmasOOV: ('asks', 'NOUN') -> ask
lemmatized via getAllLemmasOOV: ('asks', 'NOUN') -> ask
lemmatized via getAllLemmasOOV: ('explains', 'NOUN') -> explain
lemmatized via getAllLemmasOOV: ('explains', 'NOUN') -> explain
lemmatized via getAllLemmasOOV: ('tells', 'NOUN') -> tell
lemmatized via getAllLemmasOOV: ('tells', 'NOUN') -> tell
lemmatized via getAllLemmasOOV: ('texas', 'NOUN') -> texa
lemmatized via getAllLemmasOOV: ('texas', 'NOUN') -> texa
ret

retagging changed lemma for playing. Without: (playing, NOUN) -> playing, With: (playing, VERB) -> play
lemmatized via getAllLemmasOOV: ('brokes', 'VERB') -> broke
lemmatized via getAllLemmasOOV: ('brokes', 'NOUN') -> broke
lemmatized via getAllLemmasOOV: ('anyways', 'VERB') -> anyway
lemmatized via getAllLemmasOOV: ('anyways', 'NOUN') -> anyway
retagging changed lemma for selling. Without: (selling, NOUN) -> selling, With: (selling, VERB) -> sell
retagging changed lemma for breaking. Without: (breaking, NOUN) -> breaking, With: (breaking, VERB) -> break
retagging changed lemma for disgusting. Without: (disgusting, NOUN) -> disgusting, With: (disgusting, VERB) -> disgust
retagging changed lemma for playing. Without: (playing, NOUN) -> playing, With: (playing, VERB) -> play
lemmatized via getAllLemmasOOV: ('asks', 'NOUN') -> ask
lemmatized via getAllLemmasOOV: ('asks', 'NOUN') -> ask
lemmatized via getAllLemmasOOV: ('realizes', 'NOUN') -> realize
lemmatized via getAllLemmasOOV: ('realiz

lemmatized via getAllLemmasOOV: ('unfazed', 'ADJ') -> unfaze
lemmatized via getAllLemmasOOV: ('unfazed', 'ADJ') -> unfaze
lemmatized via getAllLemmasOOV: ('baking', 'ADJ') -> bake
retagging changed lemma for baking. Without: (baking, ADJ) -> bake, With: (baking, NOUN) -> baking
lemmatized via getAllLemmasOOV: ('escalates', 'NOUN') -> escalate
lemmatized via getAllLemmasOOV: ('escalates', 'NOUN') -> escalate
retagging changed lemma for playing. Without: (playing, NOUN) -> playing, With: (playing, VERB) -> play
lemmatized via getAllLemmasOOV: ('tells', 'NOUN') -> tell
lemmatized via getAllLemmasOOV: ('tells', 'NOUN') -> tell
retagging changed lemma for listening. Without: (listening, NOUN) -> listening, With: (listening, VERB) -> listen
lemmatized via getAllLemmasOOV: ('yatta', 'VERB') -> yatt
retagging changed lemma for yatta. Without: (yatta, VERB) -> yatt, With: (yatta, NOUN) -> yatta
coming -> comy instead of come
lemmatized via getAllLemmasOOV: ('coming', 'ADJ') -> comy
retagging ch

...

lemmatized via getAllLemmasOOV: ('thinks', 'NOUN') -> think
lemmatized via getAllLemmasOOV: ('thinks', 'NOUN') -> think
lemmatized via getAllLemmasOOV: ('anyways', 'VERB') -> anyway
lemmatized via getAllLemmasOOV: ('anyways', 'NOUN') -> anyway
lemmatized via getAllLemmasOOV: ('anyways', 'NOUN') -> anyway
lemmatized via getAllLemmasOOV: ('anyways', 'NOUN') -> anyway
lemmatized via getAllLemmasOOV: ('decides', 'NOUN') -> decide
lemmatized via getAllLemmasOOV: ('decides', 'NOUN') -> decide
lemmatized via getAllLemmasOOV: ('anyways', 'VERB') -> anyway
lemmatized via getAllLemmasOOV: ('anyways', 'NOUN') -> anyway
lemmatized via getAllLemmasOOV: ('anyways', 'VERB') -> anyway
lemmatized via getAllLemmasOOV: ('anyways', 'NOUN') -> anyway
lemmatized via getAllLemmasOOV: ('bluths', 'NOUN') -> bluth
lemmatized via getAllLemmasOOV: ('bluths', 'NOUN') -> bluth
lemmatized via getAllLemmasOOV: ('announces', 'NOUN') -> announce
lemmatized via getAllLemmasOOV: ('announces', 'NOUN') -> announce
retaggin

lemmatized via getAllLemmasOOV: ('hardworking', 'VERB') -> hardwork
lemmatized via getAllLemmasOOV: ('hardworking', 'VERB') -> hardwork
lemmatized via getAllLemmasOOV: ('sister', 'ADJ') -> sist
retagging changed lemma for sister. Without: (sister, ADJ) -> sist, With: (sister, NOUN) -> sister
lemmatized via getAllLemmasOOV: ('fundraisers', 'NOUN') -> fundraiser
lemmatized via getAllLemmasOOV: ('fundraisers', 'NOUN') -> fundraiser
retagging changed lemma for drumming. Without: (drumming, NOUN) -> drumming, With: (drumming, VERB) -> drum
retagging changed lemma for sleeping. Without: (sleeping, NOUN) -> sleeping, With: (sleeping, VERB) -> sleep
lemmatized via getAllLemmasOOV: ('towards', 'NOUN') -> toward
retagging changed lemma for towards. Without: (towards, None) -> towards, With: (towards, NOUN) -> toward
lemmatized via getAllLemmasOOV: ('tobias', 'VERB') -> tobia
retagging changed lemma for tobias. Without: (tobias, VERB) -> tobia, With: (tobias, NOUN) -> tobias
lemmatized via getAll

lemmatized via getAllLemmasOOV: ('buster', 'ADJ') -> bust
retagging changed lemma for buster. Without: (buster, ADJ) -> bust, With: (buster, NOUN) -> buster
lemmatized via getAllLemmasOOV: ('legs', 'VERB') -> leg
lemmatized via getAllLemmasOOV: ('tells', 'NOUN') -> tell
lemmatized via getAllLemmasOOV: ('tells', 'NOUN') -> tell
lemmatized via getAllLemmasOOV: ('sees', 'NOUN') -> see
lemmatized via getAllLemmasOOV: ('sees', 'NOUN') -> see
lemmatized via getAllLemmasOOV: ('tobias', 'VERB') -> tobia
retagging changed lemma for tobias. Without: (tobias, VERB) -> tobia, With: (tobias, NOUN) -> tobias
lemmatized via getAllLemmasOOV: ('sometimes', 'NOUN') -> sometime
retagging changed lemma for sometimes. Without: (sometimes, NOUN) -> sometime, With: (sometimes, ADV) -> sometimes
lemmatized via getAllLemmasOOV: ('shocked', 'ADJ') -> shock
retagging changed lemma for managing. Without: (managing, NOUN) -> managing, With: (managing, VERB) -> manage
lemmatized via getAllLemmasOOV: ('arrested', 'A

lemmatized via getAllLemmasOOV: ('developments', 'VERB') -> development
lemmatized via getAllLemmasOOV: ('PETA', 'ADJ') -> PETUM
retagging changed lemma for PETA. Without: (PETA, ADJ) -> petum, With: (PETA, NOUN) -> peta
lemmatized via getAllLemmasOOV: ('intertwined', 'ADJ') -> intertwine
retagging changed lemma for thinking. Without: (thinking, NOUN) -> thinking, With: (thinking, VERB) -> think
lemmatized via getAllLemmasOOV: ('stuffs', 'NOUN') -> stuff
lemmatized via getAllLemmasOOV: ('stuffs', 'NOUN') -> stuff
lemmatized via getAllLemmasOOV: ('buster', 'ADV') -> bust
retagging changed lemma for buster. Without: (buster, ADV) -> bust, With: (buster, NOUN) -> buster
lemmatized via getAllLemmasOOV: ('anyways', 'NOUN') -> anyway
lemmatized via getAllLemmasOOV: ('anyways', 'NOUN') -> anyway
lemmatized via getAllLemmasOOV: ('depressing', 'ADJ') -> depress
lemmatized via getAllLemmasOOV: ('depressing', 'ADJ') -> depress
lemmatized via getAllLemmasOOV: ('shocked', 'ADJ') -> shock
lemmatized

lemmatized via getAllLemmasOOV: ('tobias', 'ADJ') -> tobia
retagging changed lemma for tobias. Without: (tobias, ADJ) -> tobia, With: (tobias, NOUN) -> tobias
lemmatized via getAllLemmasOOV: ('thinks', 'NOUN') -> think
lemmatized via getAllLemmasOOV: ('thinks', 'NOUN') -> think
lemmatized via getAllLemmasOOV: ('themed', 'VERB') -> theme
retagging changed lemma for themed. Without: (themed, NOUN) -> themed, With: (themed, VERB) -> theme
lemmatized via getAllLemmasOOV: ('thinks', 'NOUN') -> think
lemmatized via getAllLemmasOOV: ('thinks', 'NOUN') -> think
lemmatized via getAllLemmasOOV: ('walks', 'ADJ') -> walk
lemmatized via getAllLemmasOOV: ('tells', 'NOUN') -> tell
lemmatized via getAllLemmasOOV: ('tells', 'NOUN') -> tell
lemmatized via getAllLemmasOOV: ('tobias', 'ADJ') -> tobia
retagging changed lemma for tobias. Without: (tobias, ADJ) -> tobia, With: (tobias, NOUN) -> tobias
lemmatized via getAllLemmasOOV: ('decides', 'NOUN') -> decide
lemmatized via getAllLemmasOOV: ('decides', 'N

In [504]:
LEMMA = 'much'

print(f"episodes: {EPISODE_LEMMATIZATIONS[LEMMA]}")
print(f"recalls:  {RECALL_LEMMATIZATIONS[LEMMA]}")

episodes: Counter({'much': 10})
recalls:  Counter({'much': 98})


In [461]:
getAllLemmasOOV('under-developed', 'ADJ')

{'ADJ': ('under-develop',)}

In [185]:
for word, lemma_counter in EPISODE_LEMMATIZATIONS.items():
    if word.endswith('ing'):
        print(word)
        print(lemma_counter)
        print('=============')

sitting
Counter({'sit': 26})
parking
Counter({'parking': 186})
asking
Counter({'ask': 7})
going
Counter({'go': 75})
trying
Counter({'try': 60})
talking
Counter({'talk': 36})
speaking
Counter({'speak': 14})
escalating
Counter({'escalate': 4})
staying
Counter({'stay': 10})
pointing
Counter({'point': 4})
stunting
Counter({'stunt': 2})
getting
Counter({'get': 25})
experiencing
Counter({'experience': 2})
waiting
Counter({'wait': 245})
firing
Counter({'fire': 1})
surrounding
Counter({'surround': 2})
cooking
Counter({'cook': 1})
bring
Counter({'bring': 5})
eating
Counter({'eat': 2})
playing
Counter({'play': 39})
wrapping
Counter({'wrap': 1})
trapping
Counter({'trapping': 1})
rapping
Counter({'rap': 3})
blowing
Counter({'blow': 1})
ceiling
Counter({'ceiling': 5})
listening
Counter({'listen': 7})
laying
Counter({'lay': 5})
swimming
Counter({'swim': 4})
saying
Counter({'say': 18})
meaning
Counter({'meaning': 2, 'mean': 2})
unappealing
Counter({'unappeal': 1, 'unappealing': 1})
interesting
Counte

In [131]:
RECALL_LEMMATIZATIONS['confusing']

Counter({'confuse': 12})

In [96]:
EPISODE_LEMMATIZATIONS['later']

Counter({'later': 5, 'late': 1})

In [98]:
RECALL_LEMMATIZATIONS['later']

Counter({'later': 105, 'late': 2})

In [95]:
print(f'{len(tuple(v for v in RECALL_LEMMATIZATIONS.values() if len(v) > 1))} inconsistencies (out of {len(RECALL_LEMMATIZATIONS)}):\n')
for word, lemma_counter in RECALL_LEMMATIZATIONS.items():
    if len(lemma_counter) > 1:
        print(f'\x1b[1m{word}\x1b[0m')
        for lemma, count in lemma_counter.items():
            print(f'    {lemma}    {count}')
        print('======================')

81 inconsistencies (out of 3975):

shooting
    shoot    47
    shooting    61
later
    later    105
    late    2
atlanta
    atlanta    21
    atlantum    43
swimming
    swim    30
    swimming    9
uninteresting
    uninteresting    4
    uninterest    3
left
    leave    55
    left    9
thought
    think    33
    thought    6
drinking
    drink    21
    drinking    6
smoking
    smoke    67
    smoking    23
saw
    see    68
    saw    2
heard
    hear    40
    heard    6
living
    live    20
    living    13
played
    play    62
    played    1
hundred
    hundred    68
    hundre    10
meant
    mean    5
    meant    4
feeling
    feel    15
    feeling    9
better
    good    10
    well    15
    better    1
taking
    take    51
    taking    1
opening
    opening    32
    open    5
closing
    closing    3
    close    1
beginning
    beginning    140
    begin    18
forgot
    forget    32
    forgot    1
shot
    shoot    42
    shot    57
walks
    walk    166
 

In [22]:
'well' in STOP_WORDS

False

In [18]:
find_in_recalls(r'\bafterwards', preprocess=False, show_subids=True)

In [11]:
# unchanged = []
# unfound = []
# treebank_tag = 'NNP'
# for word in set(ALL_TAGS_FOUND[treebank_tag]):
#     lemma = getAllLemmas(word, TREEBANK_LEMMINFLECT_MAPPING.get(treebank_tag))
#     if len(lemma) == 0:
#         print(f'\033[31munfound: {word}\033[0m')
#         unfound.append(word)
#         continue
#     if len(lemma) > 1:
# #         asdf
#         if any(len(v) > 1 for v in lemma.values()):
# #         if not all(len(v) == 1 for v in lemma.values()):
#             raise RuntimeError(lemma)
#             print(f'issue with {lemma}')
        
#         unique_lemmas = set(v[0] for v in lemma.values())
        
#         if len(unique_lemmas) > 1:
#             raise RuntimeError(lemma)
#             print(f'issue with {lemma}')
        
#         lemma = tuple(unique_lemmas)[0]
#     else:
#         if len(tuple(lemma.values())[0]) != 1:
# #             raise RuntimeError(lemma)
#             lemma = tuple(lemma.values())[0]
#         else:
#             lemma = tuple(lemma.values())[0][0]
    
#     if lemma != word:
#         print(f'{word} -> {lemma}')
#     else:
#         unchanged.append(word)
        
# unchanged = set(i.lower() for i in unchanged)
# print(f'============\n{"\n".join(unchanged)}')

In [565]:
all_annot = pd.concat((
    atlep1.annotations.loc[:, 'Narrative details (external events)':'Setting'].apply(lambda row: '. '.join(row.dropna()), axis=1),
    atlep2.annotations.loc[:, 'Narrative details (external events)':'Setting'].apply(lambda row: '. '.join(row.dropna()), axis=1),
    arrdev.annotations.loc[:, 'Narrative details (external events)':'Setting'].apply(lambda row: '. '.join(row.dropna()), axis=1)
)).reset_index(drop=True)
joined = all_annot.str.replace(r'(?<=\W)\. ', ' ', regex=True)
processed_annot = joined.apply(preprocess_text)
processed_annot

0       Alfred_paper_boy is sitting in his car when a ...
1       Alfred_paper_boy and Darius open the doors and...
2       Alfred_paper_boy steps out of the car. Earn ye...
3       Alfred_paper_boy slams the car door and yells ...
4       Earn, sitting in the car, calls after Alfred_p...
                              ...                        
1444    Michael confirms they are staying. George_mich...
1445    Lindsay smiles at George_michael. Michael remi...
1446    Michael puts his arm on his son's shoulder and...
1447    Maeby picks up a card and sits down. Maeby, To...
1448    George_michael stares silently. Music starts p...
Length: 1449, dtype: object

In [10]:
# for text, v in DISAGREEMENTS.items():
#     print(text)
#     for ((word, tag), lemma_lemmatize, lemma_morphy) in v:
#         print(f'\t\033[1m{word} ({tag})\033[0m')
#         print(f'\t\033[1mlemmatize:\033[0m {lemma_lemmatize}')
#         print(f'\t\033[1mmorphy:\033[0m {lemma_morphy}')
#         print('------------')
#     print('===============')

In [16]:
find_in_recalls(r"upstair(?!s)", preprocess=False, show_subids=True)

1 matches:

atlep2
    MD-102118-A-01
        ike you can't fall asleep until you find yourself upstair in your cell like you have to stay awake s


In [17]:
# target = r"\W\."
# target = "\.\.\."
target = r'upstair(?!s)'

# joined_annots = ' '.join(processed_annot)
joined_annots = ' '.join(joined)

# target = fr'\b{target}\b'
pad = 50

substrs = []
n_matches = 0

for match in re.finditer(target, joined_annots, flags=re.IGNORECASE):
    n_matches += 1
    substrs.append(joined_annots[match.start()-pad:match.start()+pad])
    
print(f'{n_matches} matches:\n')
for substr in substrs:
    print(substr)

0 matches:



### So on one hand, cleaning punctuation before collocation detection would result in there not being punctuation in the identified bigrams/trigrams. But on the other hand, it would cause bigrams/trigrams to be improperly identified across sentence boundaries (e.g., the last word in one sentence plus the first word in another).

In [34]:
import collections
import re
import string
import pandas as pd
from nltk import pos_tag, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from num2words import num2words


from nltk.collocations import (
    BigramCollocationFinder, 
    BigramAssocMeasures, 
    TrigramCollocationFinder, 
    TrigramAssocMeasures
)
from nltk import word_tokenize
from itertools import chain

annotations = arrdev.annotations.loc[:, 'Narrative details (external events)':'Setting']

stop_words = set(stopwords.words('english'))

def prepare_for_collocations(row):
    # Only use the text columns that form full sentences
    relevant = row[['Narrative details (external events)',
                    'Narrative details (internal state)',
                    'Speech']].dropna()
    text = ' '.join(relevant)
    
    # Normalize case
    text = text.lower()
    
    # Remove punctuation (but keep contractions like don't)
    text = re.sub(f"[{re.escape(string.punctuation.replace("'", ''))}]", "", text)
    
    # Tokenize
    tokens = word_tokenize(text)
    
    # Remove stopwords and trivial short tokens
    tokens = [t for t in tokens if t not in stop_words and len(t) > 1]
    return tokens

tokenized_corpus = [prepare_for_collocations(row) for _, row in annotations.iterrows()]
flat_tokens = [tok for doc in tokenized_corpus for tok in doc]


# Find frequent bigrams
bigram_finder = BigramCollocationFinder.from_words(flat_tokens)
bigram_finder.apply_freq_filter(5)  # Only keep bigrams seen ≥5 times
# bigram_finder.apply_word_filter(lambda i: i in string.punctuation)

# Score them (PMI tends to work well)
bigrams_scored = bigram_finder.score_ngrams(BigramAssocMeasures.pmi)

# Extract the top-N as a set for merging later
top_bigrams = set(['_'.join(b) for b, score in bigrams_scored[:500]])  # adjust number as needed

# (Optionally) do the same for trigrams
trigram_finder = TrigramCollocationFinder.from_words(flat_tokens)
trigram_finder.apply_freq_filter(5)
# trigram_finder.apply_word_filter(lambda i: i in string.punctuation)
trigrams_scored = trigram_finder.score_ngrams(TrigramAssocMeasures.pmi)
top_trigrams = set(['_'.join(t) for t, score in trigrams_scored[:200]])

In [438]:
# TARGET = " yea "
# REPLACE = " didn't "
# DO_REPLACEMENT = False


# from analysis_helpers.constants import TRANSCRIPTIONS_DIR
# for p in participants:
#     for rectype in ('atlep1', 'delayed', 'atlep2', 'arrdev'):
#         if (rectype == 'arrdev' and p.condition == 'A') or (rectype == 'atlep2' and p.condition == 'B'):
#             continue
        
#         sesid = p.ses1_id if rectype == 'atlep1' else p.ses2_id
#         suffix = 'delayed' if rectype == 'delayed' else 'recall'
#         fpath = TRANSCRIPTIONS_DIR.joinpath(p.subid, sesid, f'{sesid}-{suffix}.txt')
#         transcript_from_file = fpath.read_text()
        
#         if TARGET in transcript_from_file:
#             print(fpath)
#             if DO_REPLACEMENT:
#                 updated_transcript = transcript_from_file.replace(TARGET, REPLACE)
#                 fpath.write_text(updated_transcript)

/mnt/data/raw/transcriptions/MD-022719-A-01/debugYwV6a:debugs5Yuo/debugYwV6a:debugs5Yuo-delayed.txt
